# 체형·목적 기반 AI 코디 추천

입력 검증 → 체형 분석 → 의류 분리 → DeepFashion 라벨 체계 기반 착장 분석 → 코디 추천 → 결과 저장 순서로 실행합니다.

## 0. 팀원별 로컬 경로 설정 — 먼저 이 셀만 수정하세요

절대경로와 프로젝트 폴더 기준 상대경로를 모두 사용할 수 있습니다. 빈 데이터·출력 경로는 프로젝트 내부 기본 폴더를 사용합니다.

In [ ]:
from pathlib import Path
import json
import os
import sys

from IPython.display import display
from PIL import Image

# ======================================================================
# [팀원별 로컬 설정] 아래 경로만 본인의 환경에 맞게 수정하세요.
PROJECT_DIR_INPUT = r''                 # 비워두면 현재 폴더에서 자동 탐색
IMAGE_PATH_INPUT = r'data/input_person4.jpg'  # 분석할 전신사진
DATA_DIR_INPUT = r''                    # 비워두면 PROJECT_DIR/data
RULES_PATH_INPUT = r'FASHION_RULES_MASTER.md'  # 팀 규칙을 합친 통합 패션 규칙 MD
ATTRIBUTE_HEADS_PATH_INPUT = r'models/fashion_attribute_heads.pt'  # 학습 Notebook의 출력
OUTPUT_DIR_INPUT = r''                  # 비워두면 PROJECT_DIR/outputs
FONT_PATH_INPUT = r''                   # 선택: 한글 TTF/TTC 글꼴
# ======================================================================

def resolve_local_path(value, default, base_dir):
    selected = Path(value).expanduser() if value else Path(default)
    if not selected.is_absolute():
        selected = Path(base_dir) / selected
    return selected.resolve()

if PROJECT_DIR_INPUT:
    PROJECT_DIR = Path(PROJECT_DIR_INPUT).expanduser().resolve()
else:
    candidates = [Path.cwd(), Path.cwd() / 'ai_fashion_recommender']
    PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'config.py').exists()), None)
if PROJECT_DIR is None or not (PROJECT_DIR / 'config.py').is_file():
    raise FileNotFoundError('PROJECT_DIR_INPUT에 올바른 프로젝트 폴더를 입력하세요.')

IMAGE_PATH = resolve_local_path(IMAGE_PATH_INPUT, 'data/input_person.jpg', PROJECT_DIR)
DATA_DIR = resolve_local_path(DATA_DIR_INPUT, 'data', PROJECT_DIR)
RULES_PATH = resolve_local_path(RULES_PATH_INPUT, 'FASHION_RULES_MASTER.md', PROJECT_DIR)
ATTRIBUTE_HEADS_PATH = resolve_local_path(ATTRIBUTE_HEADS_PATH_INPUT, 'models/fashion_attribute_heads.pt', PROJECT_DIR)
OUTPUT_DIR = resolve_local_path(OUTPUT_DIR_INPUT, 'outputs', PROJECT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if FONT_PATH_INPUT:
    os.environ['FASHION_FONT_PATH'] = str(resolve_local_path(FONT_PATH_INPUT, FONT_PATH_INPUT, PROJECT_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from clothing_parser import ClothingParser
from fashion_model import FashionClassifier
from feedback_store import FeedbackStore
from outfit_analyzer import OutfitAnalyzer
from pose_analyzer import PoseAnalyzer
from product_catalog import ProductCatalog
from quality_checker import QualityChecker
from recommendation_engine import RecommendationEngine
from schemas import UserProfile, WardrobeItem
from virtual_tryon import VirtualTryOnAdapter

print('프로젝트 폴더:', PROJECT_DIR)
print('입력 이미지:', IMAGE_PATH)
print('데이터 폴더:', DATA_DIR)
print('패션 규칙:', RULES_PATH)
print('학습 속성 헤드:', ATTRIBUTE_HEADS_PATH)
print('출력 폴더:', OUTPUT_DIR)

## 1. 사용자 조건과 모델 설정

In [ ]:
profile = UserProfile(
    purpose='데일리',
    desired_style='캐주얼',
    budget=160_000,
    change_scope='전체 변경',  # 현재 유지 / 상의만 변경 / 하의만 변경 / 전체 변경
    height_cm=None,
    weight_kg=None,
    season='사계절',
    silhouette_goal='자동 보정 안 함',  # 자동 보정 안 함 / 균형감 / 다리가 길어 보이게 / 허리선 강조 / 상체 강조 / 하체 강조
    dress_code='자동',  # 자동 / 캐주얼 / 스마트 캐주얼 / 비즈니스 캐주얼 / 포멀
    activity_level='보통',  # 낮음 / 보통 / 높음
    preferred_colors=[],
    avoided_colors=[],
    avoided_materials=[],
    excluded_item_types=[],
    # 상세 날씨를 모르면 None으로 둡니다. 입력한 항목에 해당하는 규칙만 적용됩니다.
    temperature_c=None,
    feels_like_c=None,
    humidity=None,
    precipitation_probability=None,
    wind_mps=None,
    uv_index=None,
    # 예: [WardrobeItem('MY-BOT-01', 'bottom', '네이비', style='캐주얼')]
    owned_items=[],
)
USE_FASHN_PARSER = True
USE_FASHION_SIGLIP = True
USE_TRAINED_ATTRIBUTE_HEADS = ATTRIBUTE_HEADS_PATH.is_file()
USE_VTON = False
SHOW_SEGMENTATION_PREVIEW = False
SHOW_DETAILED_ANALYSIS = False

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(
        f'입력 이미지가 없습니다: {IMAGE_PATH}\n'
        '0번 셀의 IMAGE_PATH_INPUT을 본인의 전신사진 경로로 변경하세요.'
    )
if not (DATA_DIR / 'products.csv').is_file():
    raise FileNotFoundError(f'상품 파일이 없습니다: {DATA_DIR / "products.csv"}')
if not RULES_PATH.is_file():
    raise FileNotFoundError(f'규칙 Markdown 파일이 없습니다: {RULES_PATH}')
display(Image.open(IMAGE_PATH).convert('RGB'))
print(profile.to_dict())

## 2. 입력 사진 품질 검사

In [ ]:
# 이 셀을 다시 실행할 때 이전 MediaPipe 인스턴스를 먼저 정리합니다.
if 'pose_analyzer' in globals():
    pose_analyzer.close()
pose_analyzer = PoseAnalyzer(model_complexity=1)
quality_checker = QualityChecker(pose_analyzer)
# 포즈는 여기서 한 번만 계산하고 이후 단계에서 재사용합니다.
pose_result = pose_analyzer.analyze(IMAGE_PATH)
input_quality = quality_checker.check_input(IMAGE_PATH, pose=pose_result)
print(json.dumps(input_quality, ensure_ascii=False, indent=2))
if not input_quality['passed']:
    raise ValueError('전신사진 품질 기준을 통과하지 못했습니다. issues 항목을 확인하세요.')

## 3. MediaPipe 체형·자세 분석

In [ ]:
if not pose_result.valid:
    raise ValueError('유효한 전신 포즈를 찾지 못했습니다.')
display(pose_analyzer.draw_landmarks(IMAGE_PATH, analysis=pose_result))
print(json.dumps({k: v for k, v in pose_result.to_dict().items() if k != 'landmarks'}, ensure_ascii=False, indent=2))

## 4. 의류 분리와 현재 착장 분석

FASHN으로 픽셀 마스크를 만들고 MediaPipe로 기장을 계산합니다. FashionSigLIP 후보군은 DeepFashion-MultiModal의 소재·패턴·네크라인 라벨에 맞춰 구성했습니다.

In [ ]:
clothing_parser = ClothingParser(use_fashn=USE_FASHN_PARSER)
fashion_classifier = FashionClassifier(
    enabled=USE_FASHION_SIGLIP,
    attribute_checkpoint=ATTRIBUTE_HEADS_PATH if USE_TRAINED_ATTRIBUTE_HEADS else None,
)
print('FashionSigLIP 실행 장치:', fashion_classifier.device)
print('학습된 다중 속성 헤드:', '사용' if fashion_classifier.trained_attributes_enabled else '미사용(제로샷 폴백)')
outfit_analyzer = OutfitAnalyzer(clothing_parser, fashion_classifier)
outfit_result, parsed = outfit_analyzer.analyze(IMAGE_PATH, pose_result)
if USE_FASHN_PARSER and parsed['backend'] != 'fashn-human-parser':
    raise RuntimeError('FASHN 파서가 요청되었지만 실제 모델이 사용되지 않았습니다.')
segmentation_preview = clothing_parser.colorize(parsed['segmentation'])
segmentation_path = OUTPUT_DIR / 'fashn_segmentation.jpg'
segmentation_preview.save(segmentation_path)
if SHOW_SEGMENTATION_PREVIEW:
    display(segmentation_preview)
outfit_summary = outfit_result.to_summary_dict()
print('\n=== 착장 분석 결론 ===')
print('상의:', outfit_summary['상의'])
print('하의:', outfit_summary['하의'])
if SHOW_DETAILED_ANALYSIS:
    print('\n[상세 분석]')
    print(json.dumps(outfit_result.to_dict(), ensure_ascii=False, indent=2))
print('의류 분할 결과:', segmentation_path)

## 5. 코디 후보 탐색과 순위 결정

In [ ]:
catalog = ProductCatalog(DATA_DIR / 'products.csv')
recommender = RecommendationEngine(RULES_PATH, catalog)
print(
    f'구현된 패션 규칙: {len(recommender.active_rule_ids)}개 / '
    f'문서 규칙: {len(recommender.documented_rule_ids)}개 '
    f'(순위 점수 규칙 {len(recommender.scoring_rule_ids)}개) / {recommender.rules_source.name}'
)
if recommender.unsupported_rule_ids:
    print('추가 데이터가 필요한 규칙:')
    for rule_id in recommender.unsupported_rule_ids:
        print(f'  - {rule_id}: {recommender.UNSUPPORTED_RULE_REASONS[rule_id]}')
recommendations = recommender.recommend(profile, pose_result, outfit_result, top_k=3)
for recommendation in recommendations:
    print(f'\n#{recommendation.rank} 총점: {recommendation.total_score:.1f}')
    for product in recommendation.products:
        print(f'  - {product.name} / {product.color} / {product.price:,}원')
    for reason in recommendation.reasons:
        print('  ·', reason)
    for tip in recommendation.styling_tips:
        print('  · 스타일링 팁:', tip)
    print('  · 적용 규칙:', ', '.join(recommendation.applied_rules) or '없음')
    print(f'  · 현재 점수 계산 범위: {recommendation.score_coverage:.0f}%')

## 6. 결과 이미지 생성

In [ ]:
tryon = VirtualTryOnAdapter(enabled=USE_VTON)
preview_path = tryon.generate(
    person_image=IMAGE_PATH,
    recommendation=recommendations[0],
    output_path=OUTPUT_DIR / 'recommendation_preview.jpg',
)
display(Image.open(preview_path))
print('결과 저장 위치:', preview_path)

## 7. 선택적 피드백 저장과 최종 요약

In [ ]:
SAVE_EXAMPLE_FEEDBACK = False
feedback_store = FeedbackStore(OUTPUT_DIR / 'feedback.jsonl')
if SAVE_EXAMPLE_FEEDBACK:
    feedback_store.append(recommendation_rank=1, action='마음에 들어요', note='테스트')

summary = {
    'input_quality_passed': input_quality['passed'],
    'body_shape': pose_result.body_shape,
    'body_shape_confidence': pose_result.body_shape_confidence,
    'parser_backend': outfit_result.parser_backend,
    'garments': {
        'upper_type': outfit_result.upper_type,
        'lower_type': outfit_result.lower_type,
        'lower_subtype': outfit_result.lower_subtype,
        'pant_leg_shape': outfit_result.pant_leg_shape,
        'pant_length': outfit_result.pant_length,
        'sleeve_length': outfit_result.sleeve_length,
        'sleeve_shape': outfit_result.sleeve_shape,
        'upper_length': outfit_result.upper_length,
        'bottom_length': outfit_result.bottom_length,
        'fit': outfit_result.fit,
        'lower_fit': outfit_result.lower_fit,
        'neckline': outfit_result.neckline,
        'pattern': outfit_result.pattern,
        'material': outfit_result.material,
        'lower_pattern': outfit_result.lower_pattern,
        'lower_material': outfit_result.lower_material,
        'collar': outfit_result.collar,
        'silhouette': outfit_result.silhouette,
        'details': outfit_result.details,
        'lower_details': outfit_result.lower_details,
        'attribute_sources': outfit_result.attribute_sources,
    },
    'best_score': recommendations[0].total_score,
    'score_coverage': recommendations[0].score_coverage,
    'implemented_rules': len(recommender.active_rule_ids),
    'unsupported_rules': recommender.unsupported_rule_ids,
    'preview_path': str(preview_path),
}
final_outfit_summary = outfit_result.to_summary_dict()
print('\n=== 최종 요약 ===')
print('상의:', final_outfit_summary['상의'])
print('하의:', final_outfit_summary['하의'])
if SHOW_DETAILED_ANALYSIS:
    print(json.dumps(summary, ensure_ascii=False, indent=2))
pose_analyzer.close()